# MOLM Publication Figure Generator

This notebook regenerates the manuscript and supplementary figures from the finalized experiment-result bundles. It validates the expected architecture/configuration metadata when the corresponding JSON locks are available and keeps large optional raw-prediction files non-essential for figure generation.


In [ ]:
from pathlib import Path
import os, json, zipfile, hashlib, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

if Path("/kaggle/input").exists():
    INPUT_ROOT = Path("/kaggle/input")
    WORK_ROOT = Path("/kaggle/working/MOLM_Publication_Figures_v9")
else:
    INPUT_ROOT = Path("/mnt/data")
    WORK_ROOT = Path("/mnt/data/MOLM_Publication_Figures_v9")

EXTRACT_ROOT = WORK_ROOT / "_extracted"
FIG_DIR = WORK_ROOT / "figures"
SOURCE_DIR = WORK_ROOT / "figure_source_data"

for d in [WORK_ROOT, EXTRACT_ROOT, FIG_DIR, SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

UNIFIED_ROOT_OVERRIDE = None
MOLM_ST_ROOT_OVERRIDE = None
COMPONENT_PARETO_ROOT_OVERRIDE = None

FULL_WIDTH = 7.2
DPI = 600

plt.rcParams.update({
    "font.size": 7.5,
    "axes.titlesize": 8.5,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "figure.titlesize": 9,
    "lines.linewidth": 1.2,
    "lines.markersize": 4.5,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

SEEDS = [42, 123, 456, 789, 2024]
FEATURE_ORDER = ["onehot", "mean_esm2", "mean_fusion"]
FEATURE_LABEL = {
    "onehot": "OneHot",
    "mean_esm2": "Mean-ESM2",
    "mean_fusion": "Mean-Fusion",
    "site_esm2": "Site-ESM2",
    "site_fusion": "Site-Fusion",
}
MODEL_ORDER = ["Standard-MOLM", "Routed-MOLM", "MOLM-ST", "NN", "LDA"]
MODEL_MARKERS = {
    "Standard-MOLM": "o",
    "Routed-MOLM": "s",
    "MOLM-ST": "^",
    "NN": "D",
    "LDA": "x",
}
MODEL_LINESTYLE = {
    "Standard-MOLM": "-",
    "Routed-MOLM": "--",
    "MOLM-ST": "-.",
    "NN": ":",
    "LDA": "-",
}

print("Input root:", INPUT_ROOT)
print("Output root:", WORK_ROOT)

## 1. Architecture/reproducibility lock

These constants come directly from the research notebooks and are used only to validate the figure annotations. This notebook does not instantiate or retrain the models.

In [ ]:
EXPECTED_CONFIG = {
    "feature_dims": {
        "onehot": 2300,
        "mean_esm2": 320,
        "mean_fusion": 2620,
        "site_esm2": 2560,
        "site_fusion": 4860,
    },
    "shared_encoder": [256, 128],
    "task_tower": [64, 32, 16],
    "dropout": 0.2,
    "ranking_weight_target": 0.3,
    "ranking_weight_ova": 0.6,
    "gap_weight_target": 0.2,
    "gap_weight_ova": 0.6,
    "ranking_margin": 0.3,
    "gap_margin": 0.2,
    "focal_gamma": 2.0,
    "routed_private_adapter_dim": 32,
    "routed_dominance_weight": 0.03,
    "routed_dominance_margin": 0.20,
    "routed_dominance_warmup_epochs": 5,
    "routed_private_gate_init": -2.0,
}

ARCH_ARM_LABELS = {
    "shared_base": "A",
    "shared_dom": "B",
    "shared_dom_pcgrad": "C",
    "shared_dom_private": "D",
    "full_routed": "E",
    "independent_st_dom": "F",
    "shared_capacity_dom": "G",
    "shared_capacity_dom_pcgrad": "H",
}

ARCH_ARM_KEY = {
    "A": "Shared base",
    "B": "Shared + dominance",
    "C": "Shared + dominance + PCGrad",
    "D": "Shared + dominance + private",
    "E": "Full Routed-MOLM",
    "F": "Independent matched-loss",
    "G": "Capacity-matched shared + dominance",
    "H": "Capacity-matched shared + dominance + PCGrad",
}

LOSS_ORDER = ["L0_focal", "LR_focal_ranking", "LG_focal_gap", "LRG_full"]
LOSS_SHORT = {
    "L0_focal": "Focal",
    "LR_focal_ranking": "+Rank",
    "LG_focal_gap": "+Gap",
    "LRG_full": "Full",
}

print(json.dumps(EXPECTED_CONFIG, indent=2))

## 2. Auto-discover and stage result bundles

The resolver searches both extracted directories and ZIP archives and chooses a coherent bundle root from a distinctive anchor file.

In [ ]:
TARGET_ANCHORS = {
    "unified": "mutation_aggregate_summary_4models.csv",
    "molm_st": "molm_st_pareto_fixed_budget_summary.csv",
    "component_pareto": "pareto_all_components_fixed_budget_summary.csv",
}

INTERESTING_BASENAMES = {
    "mutation_aggregate_summary_4models.csv",
    "mutation_representation_contrast_tests_4models.csv",
    "mutation_hamming_site_block_bootstrap_10000.csv",
    "mutation_site_metrics_mean_sd_4models.csv",
    "mutation_holdout_pairwise_tests_4models.csv",
    "external_spearman_summary_4models.csv",
    "external_raw_predictions_4models.csv.gz",
    "external_seed_consensus_scores_4models.csv.gz",
    "external_paired_spearman_bootstrap_10000_4models.csv",
    "pareto_fixed_budget_summary_4models.csv",
    "pareto_fixed_budget_by_run_4models.csv",
    "standard_latent_pca_fixed_budget_summary.csv",
    "standard_latent_pca_fixed_budget_by_run.csv",
    "component_ablation_mutation_summary.csv",
    "ranking_gap_loss_ablation_summary.csv",
    "REPRODUCIBILITY_MANIFEST_UNIFIED.json",
    "UNIFIED_ANALYSIS_LOCK.json",
    "molm_st_external_spearman_summary.csv",
    "molm_st_pareto_fixed_budget_summary.csv",
    "MOLM_ST_REPRODUCIBILITY_MANIFEST.json",
    "pareto_all_components_fixed_budget_summary.csv",
    "pareto_all_components_fixed_budget_by_run.csv",
    "pareto_all_components_external_spearman_summary.csv",
    "PARETO_COMPONENT_ABLATION_LOCK.json",
}

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def stage_relevant_archives():
    archives = sorted(INPUT_ROOT.rglob("*.zip"))
    staged = []
    for zpath in archives:
        try:
            with zipfile.ZipFile(zpath, "r") as zf:
                basenames = {Path(n).name for n in zf.namelist()}
                if not (basenames & INTERESTING_BASENAMES):
                    continue
                tag = hashlib.sha256(str(zpath).encode()).hexdigest()[:10]
                dest = EXTRACT_ROOT / f"{zpath.stem}_{tag}"
                marker = dest / ".EXTRACTED"
                if not marker.exists():
                    shutil.rmtree(dest, ignore_errors=True)
                    dest.mkdir(parents=True, exist_ok=True)
                    zf.extractall(dest)
                    marker.write_text(sha256_file(zpath))
                staged.append(dest)
                print("Staged:", zpath)
        except zipfile.BadZipFile:
            pass
    return staged

STAGED_ROOTS = stage_relevant_archives()
SEARCH_ROOTS = [INPUT_ROOT, *STAGED_ROOTS]

def find_candidates(basename):
    hits, seen = [], set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob(basename):
            rp = str(p.resolve())
            if rp not in seen:
                seen.add(rp)
                hits.append(p)
    return sorted(hits)

def choose_bundle_root(anchor, override=None):
    if override is not None:
        root = Path(override)
        if not list(root.rglob(anchor)):
            raise FileNotFoundError(f"Override {root} does not contain {anchor}")
        return root
    hits = find_candidates(anchor)
    if not hits:
        raise FileNotFoundError(
            f"Could not find {anchor}. Attach the corresponding experiment result ZIP."
        )
    candidates = []
    for h in hits:
        parent = h.parent
        score = len(list(parent.glob("*.csv*"))) + len(list(parent.glob("*.json")))
        candidates.append((score, len(str(parent)), parent))
    candidates.sort(key=lambda x: (-x[0], x[1]))
    chosen = candidates[0][2]
    if len(hits) > 1:
        print(f"Multiple copies of {anchor} found; using {chosen}")
    return chosen

UNIFIED_ROOT = choose_bundle_root(TARGET_ANCHORS["unified"], UNIFIED_ROOT_OVERRIDE)
MOLM_ST_ROOT = choose_bundle_root(TARGET_ANCHORS["molm_st"], MOLM_ST_ROOT_OVERRIDE)
COMPONENT_ROOT = choose_bundle_root(TARGET_ANCHORS["component_pareto"], COMPONENT_PARETO_ROOT_OVERRIDE)

print("Unified:", UNIFIED_ROOT)
print("MOLM-ST:", MOLM_ST_ROOT)
print("Component Pareto:", COMPONENT_ROOT)

## 3. Load and validate figure source data

In [ ]:
def read_required(root, name):
    hits = list(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f"Required file missing under {root}: {name}")
    return pd.read_csv(hits[0])

def read_optional(root, name):
    # Search the resolved bundle first, then all attached/staged inputs.
    hits = list(Path(root).rglob(name))
    if not hits:
        for search_root in SEARCH_ROOTS:
            hits = list(Path(search_root).rglob(name))
            if hits:
                break
    if not hits:
        return None
    print(f"Optional source found: {hits[0]}")
    return pd.read_csv(hits[0])

mutation_agg = read_required(UNIFIED_ROOT, "mutation_aggregate_summary_4models.csv")
repr_contrast = read_required(UNIFIED_ROOT, "mutation_representation_contrast_tests_4models.csv")
hamming = read_required(UNIFIED_ROOT, "mutation_hamming_site_block_bootstrap_10000.csv")
external4 = read_required(UNIFIED_ROOT, "external_spearman_summary_4models.csv")

# Large raw per-sequence predictions are OPTIONAL in v4.
external_raw = read_optional(UNIFIED_ROOT, "external_raw_predictions_4models.csv.gz")
external_consensus = read_optional(UNIFIED_ROOT, "external_seed_consensus_scores_4models.csv.gz")

pareto4 = read_required(UNIFIED_ROOT, "pareto_fixed_budget_summary_4models.csv")
latent = read_required(UNIFIED_ROOT, "standard_latent_pca_fixed_budget_summary.csv")
latent_run = read_required(UNIFIED_ROOT, "standard_latent_pca_fixed_budget_by_run.csv")
component_mut = read_required(UNIFIED_ROOT, "component_ablation_mutation_summary.csv")
loss_mut = read_required(UNIFIED_ROOT, "ranking_gap_loss_ablation_summary.csv")

st_external = read_required(MOLM_ST_ROOT, "molm_st_external_spearman_summary.csv")
st_pareto = read_required(MOLM_ST_ROOT, "molm_st_pareto_fixed_budget_summary.csv")
component_pareto = read_required(COMPONENT_ROOT, "pareto_all_components_fixed_budget_summary.csv")

assert set(FEATURE_ORDER).issubset(set(mutation_agg["feature"].unique()))
assert {"Standard-MOLM", "Routed-MOLM", "NN", "LDA"}.issubset(set(mutation_agg["model"].unique()))
assert set(hamming["distance_bin"].unique()).issuperset({"d=1", "d=2", "d>=3"})
assert set(st_external["model"].unique()) == {"MOLM-ST"}
assert set(st_pareto["model"].unique()) == {"MOLM-ST"}
assert {"architecture_8arm", "loss_ablation"}.issubset(set(component_pareto["suite"].unique()))

external5 = pd.concat([external4, st_external], ignore_index=True, sort=False)
pareto5 = pd.concat([pareto4, st_pareto], ignore_index=True, sort=False)

# Determine diagnostic mode without failing the notebook.
if external_raw is not None:
    raw_check = external_raw[
        (external_raw["model"] == "Standard-MOLM") &
        (external_raw["dataset"] == "ISO") &
        (external_raw["feature"] == "onehot")
    ]
    if len(raw_check) and {"pca64_aff", "pca64_ova"}.issubset(raw_check.columns):
        DIAGNOSTIC_MODE = "raw_seed_level"
    elif external_consensus is not None:
        DIAGNOSTIC_MODE = "seed_consensus"
    else:
        DIAGNOSTIC_MODE = "summary_only"
elif external_consensus is not None:
    DIAGNOSTIC_MODE = "seed_consensus"
else:
    DIAGNOSTIC_MODE = "summary_only"

# Figure source exports.
mutation_agg.to_csv(SOURCE_DIR / "Figure2_mutation_aggregate.csv", index=False)
repr_contrast.to_csv(SOURCE_DIR / "Figure2_site_vs_mean_ESM2.csv", index=False)
hamming.to_csv(SOURCE_DIR / "Figure2_hamming.csv", index=False)
external5.to_csv(SOURCE_DIR / "Figure3_external_5models.csv", index=False)
pareto5.to_csv(SOURCE_DIR / "Figure3_pareto_5models.csv", index=False)
loss_mut.to_csv(SOURCE_DIR / "Supplementary_Figure_S2_loss_mutation.csv", index=False)
component_mut.to_csv(SOURCE_DIR / "Supplementary_Figure_S2_component_mutation.csv", index=False)
component_pareto.to_csv(SOURCE_DIR / "Supplementary_Figure_S2_component_pareto.csv", index=False)
latent.to_csv(SOURCE_DIR / "Figure4_and_S2_latent_pca.csv", index=False)

if external_raw is not None:
    external_raw.to_csv(
        SOURCE_DIR / "Figure4_external_raw_available.csv.gz",
        index=False, compression="gzip"
    )
if external_consensus is not None:
    external_consensus.to_csv(
        SOURCE_DIR / "Figure4_external_seed_consensus_available.csv.gz",
        index=False, compression="gzip"
    )

print("Loaded source tables successfully.")
print("External models:", sorted(external5["model"].dropna().unique()))
print("Pareto datasets:", sorted(pareto5["dataset"].dropna().unique()))
print("Figure 4 diagnostic mode:", DIAGNOSTIC_MODE)

if DIAGNOSTIC_MODE == "seed_consensus":
    print(
        "NOTE: raw seed-level external predictions are absent. "
        "Figure 4 will use seed-consensus sequence scores and the "
        "cross-seed latent-PCA robustness summary instead of a per-sequence PCA64 scatter."
    )
elif DIAGNOSTIC_MODE == "summary_only":
    print(
        "NOTE: neither raw nor consensus per-sequence external scores are available. "
        "Figure 4 will use aggregate summaries only."
    )

### Architecture/config validation against experiment locks

When the result bundles include their JSON locks, the notebook verifies the architecture dimensions, seeds, feature dimensions, and Routed-MOLM component settings before any figure is produced.

In [ ]:
def load_json_optional(root, name):
    hits = list(Path(root).rglob(name))
    if not hits:
        return None
    return json.loads(hits[0].read_text(encoding="utf-8"))

unified_lock = load_json_optional(UNIFIED_ROOT, "UNIFIED_ANALYSIS_LOCK.json")
component_lock = load_json_optional(COMPONENT_ROOT, "PARETO_COMPONENT_ABLATION_LOCK.json")

if unified_lock is not None:
    assert list(unified_lock["seeds"]) == SEEDS
    assert unified_lock["features"] == EXPECTED_CONFIG["feature_dims"]

    standard = unified_lock["standard_molm"]
    assert standard["shared_dims"] == EXPECTED_CONFIG["shared_encoder"]
    assert standard["tower_dims"] == EXPECTED_CONFIG["task_tower"][:2]
    assert int(standard["latent_dim"]) == EXPECTED_CONFIG["task_tower"][2]
    assert np.isclose(float(standard["dropout"]), EXPECTED_CONFIG["dropout"])
    assert standard["dominance"] is False
    assert standard["pcgrad"] is False
    assert standard["private_adapters"] is False

    print("PASS: Standard-MOLM architecture matches unified analysis lock.")
else:
    print("WARNING: UNIFIED_ANALYSIS_LOCK.json not found; CSV structure checks still apply.")

if component_lock is not None:
    comp = component_lock["architecture_component_settings"]
    assert int(comp["PRIVATE_ADAPTER_DIM"]) == EXPECTED_CONFIG["routed_private_adapter_dim"]
    assert np.isclose(float(comp["PRIVATE_GATE_INIT"]), EXPECTED_CONFIG["routed_private_gate_init"])
    assert np.isclose(float(comp["DOMINANCE_WEIGHT"]), EXPECTED_CONFIG["routed_dominance_weight"])
    assert np.isclose(float(comp["DOMINANCE_MARGIN"]), EXPECTED_CONFIG["routed_dominance_margin"])
    assert int(comp["DOMINANCE_WARMUP"]) == EXPECTED_CONFIG["routed_dominance_warmup_epochs"]

    losses = component_lock["loss_settings"]
    assert np.isclose(float(losses["RANKING_WEIGHT_AFF"]), EXPECTED_CONFIG["ranking_weight_target"])
    assert np.isclose(float(losses["RANKING_WEIGHT_SPEC"]), EXPECTED_CONFIG["ranking_weight_ova"])
    assert np.isclose(float(losses["GAP_WEIGHT_AFF"]), EXPECTED_CONFIG["gap_weight_target"])
    assert np.isclose(float(losses["GAP_WEIGHT_SPEC"]), EXPECTED_CONFIG["gap_weight_ova"])
    assert np.isclose(float(losses["RANKING_MARGIN"]), EXPECTED_CONFIG["ranking_margin"])
    assert np.isclose(float(losses["GAP_MARGIN"]), EXPECTED_CONFIG["gap_margin"])
    assert np.isclose(float(losses["FOCAL_GAMMA"]), EXPECTED_CONFIG["focal_gamma"])

    print("PASS: Routed-MOLM and loss settings match component experiment lock.")
else:
    print("WARNING: PARETO_COMPONENT_ABLATION_LOCK.json not found; CSV structure checks still apply.")

## 4. Shared plotting helpers

In [ ]:
def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=3, width=0.7)

def panel_label(ax, letter):
    # Slightly outside the upper-left corner; bbox_inches="tight" preserves it.
    ax.text(-0.08, 1.04, letter, transform=ax.transAxes,
            fontweight="bold", fontsize=9, va="top", ha="left")

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=DPI, bbox_inches="tight")
    print("Saved:", pdf)
    print("Saved:", png)
    return pdf, png

def model_errorbar(ax, x, y, yerr, model, label=None):
    ax.errorbar(
        x, y, yerr=yerr,
        marker=MODEL_MARKERS.get(model, "o"),
        linestyle=MODEL_LINESTYLE.get(model, "-"),
        capsize=2,
        label=label or model,
    )

def finite_err(x):
    arr = np.asarray(x, float)
    return np.where(np.isfinite(arr), arr, 0.0)

def annotate_significance(ax, x, y, p):
    if np.isfinite(p) and p < 0.05:
        ax.text(x, y, "*", ha="center", va="bottom", fontsize=9)

def feature_axis(ax):
    ax.set_xticks(range(len(FEATURE_ORDER)))
    ax.set_xticklabels([FEATURE_LABEL[f] for f in FEATURE_ORDER],
                       rotation=15, ha="right")

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def minmax(x):
    x = np.asarray(x, float)
    lo, hi = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(hi - lo) or hi <= lo:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)

# Exact fixed-budget Pareto helpers used by the experiment notebook.
def pareto_mask_max(points):
    points = np.asarray(points, float)
    mask = np.ones(len(points), bool)
    for i in range(len(points)):
        if (np.all(points >= points[i], axis=1) &
            np.any(points > points[i], axis=1)).any():
            mask[i] = False
    return mask

def nondominated_sort(points):
    remaining = np.arange(len(points))
    fronts = []
    while len(remaining):
        mask = pareto_mask_max(points[remaining])
        fronts.append(remaining[mask])
        remaining = remaining[~mask]
    return fronts

def crowding_distance(points):
    points = np.asarray(points, float)
    d = np.zeros(len(points), float)
    if len(points) <= 2:
        d[:] = np.inf
        return d
    for obj in range(points.shape[1]):
        order = np.argsort(points[:, obj], kind="mergesort")
        d[order[0]] = d[order[-1]] = np.inf
        span = points[order[-1], obj] - points[order[0], obj]
        if span <= 0:
            continue
        for r in range(1, len(points)-1):
            cur = order[r]
            if np.isfinite(d[cur]):
                d[cur] += (
                    points[order[r+1], obj] -
                    points[order[r-1], obj]
                ) / span
    return d

def select_fixed_budget(points, identifiers, k):
    points = np.asarray(points, float)
    ids = np.asarray(identifiers).astype(str)
    selected = []
    for front in nondominated_sort(points):
        if len(selected) + len(front) <= k:
            selected.extend(front.tolist())
            continue
        rem = k - len(selected)
        dist = crowding_distance(points[front])
        order = sorted(
            range(len(front)),
            key=lambda j: (-dist[j], ids[front[j]])
        )
        selected.extend(front[order[:rem]].tolist())
        break
    return np.asarray(selected, int)

def normalize_true_objectives(target, ova):
    p = np.c_[np.asarray(target, float), -np.asarray(ova, float)]
    mn = p.min(axis=0)
    sp = p.max(axis=0) - mn
    sp[sp == 0] = 1.0
    return (p - mn) / sp

def hypervolume_2d_max(points):
    points = np.asarray(points, float)
    points = points[np.all(points >= 0, axis=1)]
    if not len(points):
        return 0.0
    points = points[pareto_mask_max(points)]
    points = points[np.argsort(points[:, 0])]
    total, prev = 0.0, 0.0
    for x, y in points:
        total += max(0.0, x - prev) * max(0.0, y)
        prev = max(prev, x)
    return float(total)

def igd(true_front, selected):
    true_front = np.asarray(true_front, float)
    selected = np.asarray(selected, float)
    return float(
        np.sqrt(
            ((true_front[:, None, :] - selected[None, :, :])**2).sum(axis=2)
        ).min(axis=1).mean()
    )

def fixed_budget_metrics(g, selected_idx):
    true_obj = normalize_true_objectives(g["true_aff"], g["true_ova"])
    true_mask = g["is_true_pareto"].astype(bool).to_numpy()
    n_true = int(true_mask.sum())
    hits = int(true_mask[selected_idx].sum())
    k = len(selected_idx)
    prevalence = n_true / len(g)
    recall = hits / n_true if n_true else np.nan
    precision = hits / k if k else np.nan
    enrichment = precision / prevalence if prevalence > 0 else np.nan
    hv = hypervolume_2d_max(true_obj[selected_idx])
    igd_value = igd(true_obj[true_mask], true_obj[selected_idx])
    return {
        "recall": recall,
        "precision": precision,
        "enrichment": enrichment,
        "hv": hv,
        "igd": igd_value,
        "hits": hits,
        "k": k,
    }

# Main Figure 1 — Keep the existing manuscript architecture figure

**This notebook intentionally does not regenerate Figure 1.**

The current manuscript's architecture artwork should remain Figure 1.  
The plotting notebook starts generating new artwork at **Figure 2**.

If the old artwork still contains `Affinity` / `Specificity` labels inside the image itself, update those visible labels separately to `Target-binding proxy` / `OVA-binding proxy` while preserving the architecture layout.


In [ ]:
print("Figure 1 is intentionally preserved from the manuscript; no replacement file is generated.")

# Main Figure 2 — Mutation generalization

A cleaner, page-efficient six-panel figure:

- **A/B:** primary top-residue mutation-holdout MCC.
- **C/D:** Site-ESM2 versus Mean-ESM2.
- **E/F:** Hamming-distance degradation.

The model legend is shared above the figure rather than repeated inside panels.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8.8, 5.7))
fig.subplots_adjust(
    left=0.08, right=0.985, top=0.94, bottom=0.16,
    wspace=0.34, hspace=0.46
)

# A/B — primary mutation-holdout MCC
for col, task in enumerate(["affinity", "ova"]):
    ax = axes[0, col]
    sub = mutation_agg[
        (mutation_agg["analysis_role"] == "primary") &
        (mutation_agg["holdout_type"] == "top") &
        (mutation_agg["task"] == task) &
        (mutation_agg["feature"].isin(FEATURE_ORDER))
    ]
    for model in ["Standard-MOLM", "Routed-MOLM", "NN", "LDA"]:
        g = (
            sub[sub["model"] == model]
            .set_index("feature")
            .reindex(FEATURE_ORDER)
        )
        model_errorbar(
            ax,
            np.arange(len(FEATURE_ORDER)),
            g["site_mean_mcc"].to_numpy(float),
            finite_err(g["site_sd_mcc"]),
            model,
        )

    feature_axis(ax)
    ax.set_ylabel("MCC")
    letter = "A" if task == "affinity" else "B"
    label = "Target proxy" if task == "affinity" else "OVA proxy"
    ax.set_title(f"{letter}  {label}", pad=8)
    clean_axes(ax)

# C/D — residue-focused ESM-2 contrast
for col, task in enumerate(["affinity", "ova"]):
    ax = axes[1, col]
    sub = repr_contrast[
        (repr_contrast["analysis_role"] == "primary") &
        (repr_contrast["holdout_type"] == "top") &
        (repr_contrast["task"] == task) &
        (repr_contrast["representation_contrast"]
         .astype(str).str.contains("Site-ESM2"))
    ].copy()

    order = ["Standard-MOLM", "Routed-MOLM", "NN", "LDA"]
    sub["order"] = sub["model"].map({m: j for j, m in enumerate(order)})
    sub = sub.sort_values("order")
    x = np.arange(len(sub))

    ax.axhline(0, linewidth=0.8, linestyle="--")
    ax.plot(x, sub["mean_delta_mcc"], marker="o", linestyle="")
    for xx, yy, pp in zip(
        x,
        sub["mean_delta_mcc"],
        sub["wilcoxon_p_holm_2_representation_contrasts"],
    ):
        annotate_significance(ax, xx, yy, pp)

    ax.set_xticks(x)
    ax.set_xticklabels(["Standard", "Routed", "NN", "LDA"], rotation=12, ha="right")
    ax.set_ylabel("ΔMCC: Site − Mean ESM-2")
    letter = "C" if task == "affinity" else "D"
    label = "Target proxy" if task == "affinity" else "OVA proxy"
    ax.set_title(f"{letter}  {label}", pad=8)
    clean_axes(ax)

# E/F — Hamming-distance stratification, Mean-ESM2
distance_order = ["d=1", "d=2", "d>=3"]
for row_idx, task in enumerate(["affinity", "ova"]):
    ax = axes[row_idx, 2]
    sub = hamming[
        (hamming["feature"] == "mean_esm2") &
        (hamming["task"] == task) &
        (hamming["metric"] == "mcc")
    ]

    for model in ["Standard-MOLM", "Routed-MOLM", "NN", "LDA"]:
        g = (
            sub[sub["model"] == model]
            .set_index("distance_bin")
            .reindex(distance_order)
        )
        y = g["mean"].to_numpy(float)
        lo = g["ci_low"].to_numpy(float)
        hi = g["ci_high"].to_numpy(float)
        yerr = np.vstack([y - lo, hi - y])
        ax.errorbar(
            np.arange(3), y, yerr=yerr,
            marker=MODEL_MARKERS.get(model, "o"),
            linestyle=MODEL_LINESTYLE.get(model, "-"),
            capsize=2,
            label=model,
        )

    ax.set_xticks(range(3))
    ax.set_xticklabels(["1", "2", "≥3"])
    ax.set_xlabel("Minimum Hamming distance")
    ax.set_ylabel("MCC")
    if task == "affinity":
        ax.set_title("E  Target · Mean-ESM2", pad=8)
    else:
        ax.set_title("F  OVA · Mean-ESM2", pad=8)
    clean_axes(ax)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=4,
    frameon=False,
)

save_figure(fig, "Figure2_Mutation_Generalization_Clean")
plt.show()

# Main Figure 3 — Cross-platform transfer and fixed-budget Pareto prioritization

The external-transfer panels use a single shared legend. The right column shows the two most interpretable fixed-budget Pareto summaries for ISO Mean-Fusion.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8.8, 5.7))
fig.subplots_adjust(
    left=0.08, right=0.985, top=0.94, bottom=0.16,
    wspace=0.34, hspace=0.46
)

external_panels = [
    ("ISO", "aff", "A", "ISO target transfer"),
    ("ISO", "ova", "B", "ISO OVA transfer"),
    ("IgG-primary42", "aff", "C", "IgG-42 target transfer"),
    ("IgG-primary42", "ova", "D", "IgG-42 OVA transfer"),
]

for panel_i, (dataset, task_short, letter, title) in enumerate(external_panels):
    ax = axes[panel_i // 2, panel_i % 2]
    mean_col = f"{task_short}_spearman_mean"
    sd_col = f"{task_short}_spearman_sd"
    sub = external5[
        (external5["dataset"] == dataset) &
        (external5["feature"].isin(FEATURE_ORDER))
    ]

    for model in MODEL_ORDER:
        g = (
            sub[sub["model"] == model]
            .set_index("feature")
            .reindex(FEATURE_ORDER)
        )
        if g[mean_col].notna().sum() == 0:
            continue
        model_errorbar(
            ax,
            np.arange(len(FEATURE_ORDER)),
            g[mean_col].to_numpy(float),
            finite_err(g[sd_col]) if sd_col in g.columns else np.zeros(3),
            model,
        )

    feature_axis(ax)
    ax.set_ylabel("Spearman ρ")
    ax.set_title(f"{letter}  {title}", pad=8)
    clean_axes(ax)

# E/F — ISO Mean-Fusion fixed-budget curves
for row_idx, (metric, ylabel, letter, title) in enumerate([
    ("recall_mean", "Pareto Recall@K", "E", "ISO Pareto recall"),
    ("igd_mean", "IGD", "F", "ISO Pareto IGD"),
]):
    ax = axes[row_idx, 2]
    sub = pareto5[
        (pareto5["dataset"] == "ISO") &
        (pareto5["feature"] == "mean_fusion")
    ]

    for model in MODEL_ORDER:
        g = sub[sub["model"] == model].sort_values("k")
        if g.empty:
            continue
        err_col = "recall_sd" if metric == "recall_mean" else "igd_sd"
        ax.errorbar(
            g["k"], g[metric],
            yerr=finite_err(g[err_col]) if err_col in g.columns else None,
            marker=MODEL_MARKERS.get(model, "o"),
            linestyle=MODEL_LINESTYLE.get(model, "-"),
            capsize=2,
            label=model,
        )

    ax.set_xlabel("Screening budget K")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{letter}  {title}", pad=8)
    clean_axes(ax)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=5,
    frameon=False,
)

save_figure(fig, "Figure3_External_Transfer_and_Pareto_Clean")
plt.show()

# Main Figure 4 — Updated Pareto diagnostic figure

This reuses the **concept of the earlier manuscript diagnostic**, but it is made robust to the exact contents of the Kaggle result bundle.

### Automatic behavior
- If `external_raw_predictions_4models.csv.gz` exists: full seed-42 logit/probability/PCA64 diagnostic.
- If that large file is absent but `external_seed_consensus_scores_4models.csv.gz` exists: consensus logit/probability/measured-space diagnostic plus the current cross-seed PCA64 robustness curve.
- If both per-sequence files are absent: a summary-only diagnostic is produced from the fixed-budget and external summary tables.

This fallback does **not invent latent coordinates**. If raw PCA64 sequence scores are unavailable, the notebook plots the actual cross-seed latent-PCA performance summary instead.


In [ ]:
DIAGNOSTIC_SEED = 42
DIAGNOSTIC_K = 20

def compute_true_pareto_from_measurements(df):
    pts = np.c_[df["true_aff"].to_numpy(float), -df["true_ova"].to_numpy(float)]
    return pareto_mask_max(pts)

def latent_summary_panel(ax, metric, err_col, ylabel, title):
    sub = latent[
        (latent["dataset"] == "ISO") &
        (latent["feature"] == "onehot")
    ].copy()
    order = ["logits", "pca16", "pca32", "pca64"]
    labels = {"logits":"Logits", "pca16":"PCA16", "pca32":"PCA32", "pca64":"PCA64"}
    for space in order:
        q = sub[sub["score_space"] == space].sort_values("k")
        if q.empty:
            continue
        ax.errorbar(
            q["k"], q[metric],
            yerr=finite_err(q[err_col]) if err_col in q.columns else None,
            marker="o", linestyle="-", capsize=2, label=labels[space]
        )
    ax.set_xlabel("Screening budget K")
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    clean_axes(ax)

def latent_seed_delta_panel(ax, metric_col, ylabel, title, higher_is_better=True):
    q = latent_run[
        (latent_run["dataset"] == "ISO") &
        (latent_run["feature"] == "onehot") &
        (latent_run["score_space"].isin(["logits", "pca64"]))
    ].copy()

    # Standardize metric column names from the seed-level file.
    possible = {
        "recall": ["recall_at_k", "recall"],
        "hypervolume": ["hypervolume_true_selected", "hypervolume"],
        "igd": ["igd_true_front_to_selected", "igd"],
    }
    chosen = None
    for c in possible[metric_col]:
        if c in q.columns:
            chosen = c
            break
    if chosen is None:
        raise RuntimeError(f"No seed-level column found for {metric_col}: {possible[metric_col]}")

    piv = q.pivot_table(index=["seed", "k"], columns="score_space", values=chosen, aggfunc="first").reset_index()
    piv = piv.dropna(subset=["logits", "pca64"]).copy()
    piv["delta"] = piv["pca64"] - piv["logits"]
    if not higher_is_better:
        # Positive means PCA64 is better for every delta panel.
        piv["delta"] = -piv["delta"]

    for seed in sorted(piv["seed"].unique()):
        g = piv[piv["seed"] == seed].sort_values("k")
        ax.plot(g["k"], g["delta"], marker="o", linewidth=0.9, alpha=0.65)
    mean = piv.groupby("k", as_index=False)["delta"].mean()
    ax.plot(mean["k"], mean["delta"], marker="o", linewidth=2.0, label="Mean Δ")
    ax.axhline(0, linestyle="--", linewidth=0.8)
    ax.set_xlabel("Screening budget K")
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    clean_axes(ax)

# ================================================================
# MODE 1: raw seed-level sequence scores available
# ================================================================
if DIAGNOSTIC_MODE == "raw_seed_level":
    g = external_raw[
        (external_raw["model"] == "Standard-MOLM") &
        (external_raw["dataset"] == "ISO") &
        (external_raw["feature"] == "onehot") &
        (external_raw["seed"] == DIAGNOSTIC_SEED)
    ].sort_values("row_id").reset_index(drop=True).copy()

    required = ["true_aff","true_ova","pred_aff","pred_ova","pca64_aff","pca64_ova","sequence_id"]
    missing = [c for c in required if c not in g.columns]
    if missing:
        raise RuntimeError(f"Raw Figure 4 source missing columns: {missing}")

    if "is_true_pareto" in g.columns:
        true_mask = g["is_true_pareto"].astype(bool).to_numpy()
    else:
        true_mask = compute_true_pareto_from_measurements(g)
        g["is_true_pareto"] = true_mask

    logit_points = np.c_[g["pred_aff"], -g["pred_ova"]]
    pca64_points = np.c_[g["pca64_aff"], -g["pca64_ova"]]
    sel_logit = select_fixed_budget(logit_points, g["sequence_id"], DIAGNOSTIC_K)
    sel_pca64 = select_fixed_budget(pca64_points, g["sequence_id"], DIAGNOSTIC_K)
    m_logit = fixed_budget_metrics(g, sel_logit)
    m_pca64 = fixed_budget_metrics(g, sel_pca64)

    diag_export = g.copy()
    diag_export["selected_logit_k20"] = False
    diag_export["selected_pca64_k20"] = False
    diag_export.loc[sel_logit, "selected_logit_k20"] = True
    diag_export.loc[sel_pca64, "selected_pca64_k20"] = True
    diag_export.to_csv(SOURCE_DIR / "Figure4_raw_seed42_fixed_budget_selection.csv", index=False)

    fig, axes = plt.subplots(2, 3, figsize=(9.2, 5.8))
    fig.subplots_adjust(left=0.075, right=0.985, top=0.92, bottom=0.15, wspace=0.36, hspace=0.48)

    ax = axes[0,0]
    all_pts = ax.scatter(g["pred_aff"], g["pred_ova"], s=12, alpha=0.40, label="All ISO")
    pareto_pts = ax.scatter(g.loc[true_mask,"pred_aff"], g.loc[true_mask,"pred_ova"], s=32, marker="D", label="Measured Pareto")
    logit_pts = ax.scatter(g.loc[sel_logit,"pred_aff"], g.loc[sel_logit,"pred_ova"], s=38, marker="o", label="Logit K=20")
    ax.invert_yaxis()
    ax.set_xlabel("Target logit"); ax.set_ylabel("OVA logit ↓")
    ax.set_title("A  Output logit space", pad=8); clean_axes(ax)

    ax = axes[0,1]
    p_target, p_ova = sigmoid(g["pred_aff"]), sigmoid(g["pred_ova"])
    ax.scatter(p_target, p_ova, s=12, alpha=0.40)
    ax.scatter(p_target[true_mask], p_ova[true_mask], s=32, marker="D")
    ax.scatter(p_target[sel_logit], p_ova[sel_logit], s=38, marker="o")
    ax.invert_yaxis()
    ax.set_xlabel("Target probability"); ax.set_ylabel("OVA probability ↓")
    ax.set_title("B  Probability space", pad=8); clean_axes(ax)

    ax = axes[0,2]
    ax.scatter(g["pca64_aff"], g["pca64_ova"], s=12, alpha=0.40)
    ax.scatter(g.loc[true_mask,"pca64_aff"], g.loc[true_mask,"pca64_ova"], s=32, marker="D")
    pca_pts = ax.scatter(g.loc[sel_pca64,"pca64_aff"], g.loc[sel_pca64,"pca64_ova"], s=40, marker="s", label="PCA64 K=20")
    ax.invert_yaxis()
    ax.set_xlabel("Target PCA64"); ax.set_ylabel("OVA PCA64 ↓")
    ax.set_title("C  Latent PCA64 space", pad=8); clean_axes(ax)

    ax = axes[1,0]
    ax.hist(g["pred_aff"], bins=22, alpha=0.55, density=True, label="Target logit")
    ax.hist(g["pred_ova"], bins=22, alpha=0.55, density=True, label="OVA logit")
    ax.set_xlabel("Output logit"); ax.set_ylabel("Density")
    ax.set_title("D  Score distributions", pad=8)
    ax.legend(frameon=False, fontsize=6); clean_axes(ax)

    ax = axes[1,1]
    ax.scatter(minmax(g["true_aff"]), minmax(g["pred_aff"]), s=13, alpha=0.50, marker="o", label="Target")
    ax.scatter(minmax(-g["true_ova"]), minmax(-g["pred_ova"]), s=13, alpha=0.50, marker="^", label="OVA")
    ax.plot([0,1],[0,1], linestyle="--", linewidth=0.8)
    rho_target = pd.Series(g["pred_aff"]).corr(pd.Series(g["true_aff"]), method="spearman")
    rho_ova = pd.Series(g["pred_ova"]).corr(pd.Series(g["true_ova"]), method="spearman")
    ax.text(0.04,0.96,f"ρ target={rho_target:.3f}\nρ OVA={rho_ova:.3f}", transform=ax.transAxes, va="top", fontsize=6.2)
    ax.set_xlabel("Measured desirability"); ax.set_ylabel("Predicted desirability")
    ax.set_title("E  Predicted vs measured", pad=8)
    ax.legend(frameon=False, fontsize=6, loc="lower right"); clean_axes(ax)

    ax = axes[1,2]
    ax.scatter(g["true_aff"], g["true_ova"], s=12, alpha=0.32)
    ax.scatter(g.loc[true_mask,"true_aff"], g.loc[true_mask,"true_ova"], s=34, marker="D")
    ax.scatter(g.loc[sel_logit,"true_aff"], g.loc[sel_logit,"true_ova"], s=38, marker="o")
    ax.scatter(g.loc[sel_pca64,"true_aff"], g.loc[sel_pca64,"true_ova"], s=40, marker="s")
    ax.invert_yaxis()
    ax.set_xlabel("Measured target binding"); ax.set_ylabel("Measured OVA binding ↓")
    ax.set_title("F  Measured objective space", pad=8); clean_axes(ax)

    fig.legend([all_pts,pareto_pts,logit_pts,pca_pts],
               ["All ISO","Measured Pareto","Logit K=20","PCA64 K=20"],
               loc="lower center", bbox_to_anchor=(0.5,0.02), ncol=4, frameon=False)

    save_figure(fig, "Figure4_Pareto_Diagnostics_Raw")
    plt.show()

    FIG4_CAPTION = (
        "Figure 4. Pareto diagnostics for Standard-MOLM on ISO OneHot. "
        "The pre-specified seed 42 is used only for descriptive score-space visualization; "
        "cross-seed latent-PCA robustness is reported separately. "
        "(A) Output-logit space. (B) Probability space. (C) PCA64 latent score space. "
        "(D) Output-score distributions. (E) Predicted versus measured desirability. "
        "(F) Measured objective space with fixed-budget K=20 logit and PCA64 selections."
    )

# ================================================================
# MODE 2: consensus sequence scores available
# ================================================================
elif DIAGNOSTIC_MODE == "seed_consensus":
    g = external_consensus[
        (external_consensus["model"] == "Standard-MOLM") &
        (external_consensus["dataset"] == "ISO") &
        (external_consensus["feature"] == "onehot")
    ].sort_values("row_id").reset_index(drop=True).copy()

    required = ["true_aff","true_ova","pred_aff","pred_ova","sequence_id"]
    missing = [c for c in required if c not in g.columns]
    if missing:
        raise RuntimeError(f"Consensus Figure 4 source missing columns: {missing}")

    true_mask = compute_true_pareto_from_measurements(g)
    g["is_true_pareto"] = true_mask
    logit_points = np.c_[g["pred_aff"], -g["pred_ova"]]
    sel_logit = select_fixed_budget(logit_points, g["sequence_id"], DIAGNOSTIC_K)

    fig, axes = plt.subplots(2,3,figsize=(9.2,5.8))
    fig.subplots_adjust(left=0.075,right=0.985,top=0.92,bottom=0.15,wspace=0.36,hspace=0.48)

    ax=axes[0,0]
    all_pts=ax.scatter(g["pred_aff"],g["pred_ova"],s=12,alpha=0.40,label="All ISO")
    pareto_pts=ax.scatter(g.loc[true_mask,"pred_aff"],g.loc[true_mask,"pred_ova"],s=32,marker="D",label="Measured Pareto")
    logit_pts=ax.scatter(g.loc[sel_logit,"pred_aff"],g.loc[sel_logit,"pred_ova"],s=38,marker="o",label="Consensus K=20")
    ax.invert_yaxis(); ax.set_xlabel("Target consensus score"); ax.set_ylabel("OVA consensus score ↓")
    ax.set_title("A  Consensus score space",pad=8); clean_axes(ax)

    ax=axes[0,1]
    pt,po=sigmoid(g["pred_aff"]),sigmoid(g["pred_ova"])
    ax.scatter(pt,po,s=12,alpha=0.40); ax.scatter(pt[true_mask],po[true_mask],s=32,marker="D"); ax.scatter(pt[sel_logit],po[sel_logit],s=38,marker="o")
    ax.invert_yaxis(); ax.set_xlabel("Target transformed score"); ax.set_ylabel("OVA transformed score ↓")
    ax.set_title("B  Monotonic score transform",pad=8); clean_axes(ax)

    latent_summary_panel(axes[0,2],"recall_mean","recall_sd","Pareto Recall@K","C  Latent-score robustness")

    ax=axes[1,0]
    ax.hist(g["pred_aff"],bins=22,alpha=0.55,density=True,label="Target score")
    ax.hist(g["pred_ova"],bins=22,alpha=0.55,density=True,label="OVA score")
    ax.set_xlabel("Consensus score"); ax.set_ylabel("Density"); ax.set_title("D  Score distributions",pad=8)
    ax.legend(frameon=False,fontsize=6); clean_axes(ax)

    ax=axes[1,1]
    ax.scatter(minmax(g["true_aff"]),minmax(g["pred_aff"]),s=13,alpha=0.50,marker="o",label="Target")
    ax.scatter(minmax(-g["true_ova"]),minmax(-g["pred_ova"]),s=13,alpha=0.50,marker="^",label="OVA")
    ax.plot([0,1],[0,1],linestyle="--",linewidth=0.8)
    ax.set_xlabel("Measured desirability"); ax.set_ylabel("Predicted desirability")
    ax.set_title("E  Predicted vs measured",pad=8); ax.legend(frameon=False,fontsize=6,loc="lower right"); clean_axes(ax)

    ax=axes[1,2]
    ax.scatter(g["true_aff"],g["true_ova"],s=12,alpha=0.32)
    ax.scatter(g.loc[true_mask,"true_aff"],g.loc[true_mask,"true_ova"],s=34,marker="D")
    ax.scatter(g.loc[sel_logit,"true_aff"],g.loc[sel_logit,"true_ova"],s=38,marker="o")
    ax.invert_yaxis(); ax.set_xlabel("Measured target binding"); ax.set_ylabel("Measured OVA binding ↓")
    ax.set_title("F  Measured objective space",pad=8); clean_axes(ax)

    fig.legend([all_pts,pareto_pts,logit_pts],["All ISO","Measured Pareto","Consensus K=20"],
               loc="lower center",bbox_to_anchor=(0.5,0.02),ncol=3,frameon=False)
    save_figure(fig,"Figure4_Pareto_Diagnostics_ConsensusFallback")
    plt.show()

    FIG4_CAPTION = (
        "Figure 4. Pareto diagnostics for Standard-MOLM on ISO OneHot using seed-consensus scores. "
        "(A) Consensus score space with measured Pareto points and fixed-budget K=20 selections. "
        "(B) Monotonic score transform. (C) Cross-seed Recall@K robustness for logits and PCA16/PCA32/PCA64 latent scores. "
        "(D) Consensus-score distributions. (E) Predicted versus measured desirability. "
        "(F) Measured objective space for the consensus-selected candidates."
    )

# ================================================================
# MODE 3: summary-only fallback — no redundant Figure 3 panels
# ================================================================
else:
    fig, axes = plt.subplots(2,3,figsize=(9.2,5.8))
    fig.subplots_adjust(left=0.075,right=0.985,top=0.92,bottom=0.14,wspace=0.36,hspace=0.48)

    latent_summary_panel(axes[0,0],"recall_mean","recall_sd","Pareto Recall@K","A  Latent-score Recall")
    latent_summary_panel(axes[0,1],"hypervolume_mean","hypervolume_sd","Hypervolume","B  Latent-score hypervolume")
    latent_summary_panel(axes[0,2],"igd_mean","igd_sd","IGD","C  Latent-score IGD")

    latent_seed_delta_panel(axes[1,0],"recall","Δ Recall (PCA64 − logits)","D  Seed-level Recall gain",True)
    latent_seed_delta_panel(axes[1,1],"hypervolume","Δ HV (PCA64 − logits)","E  Seed-level HV gain",True)
    latent_seed_delta_panel(axes[1,2],"igd","Δ improvement in IGD","F  Seed-level IGD gain",False)

    handles, labels = axes[0,0].get_legend_handles_labels()
    fig.legend(handles,labels,loc="lower center",bbox_to_anchor=(0.5,0.02),ncol=4,frameon=False)

    save_figure(fig,"Figure4_Latent_PCA_Robustness_SummaryFallback")
    plt.show()

    FIG4_CAPTION = (
        "Figure 4. Robustness of latent-PCA scoring for Standard-MOLM on ISO OneHot. "
        "(A-C) Fixed-budget Pareto Recall@K, hypervolume, and IGD for output logits and PCA16/PCA32/PCA64 latent scores, "
        "reported across screening budgets. Error bars show optimization-seed variability. "
        "(D-F) Seed-level changes for PCA64 relative to logits in Recall, hypervolume, and IGD; "
        "positive values indicate improvement by PCA64. "
        "This summary-only diagnostic is used when per-sequence latent coordinates are unavailable and avoids duplicating the external-transfer panels shown in Figure 3."
    )

print("Figure 4 caption selected for mode:", DIAGNOSTIC_MODE)
print(FIG4_CAPTION)

# Supplementary Figure S1 — IgG-all96 fixed-budget Pareto

The second favorable Routed-MOLM regime remains in the Supplement, with a shared legend and compact labels.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8.8, 2.8))
fig.subplots_adjust(
    left=0.08, right=0.985, top=0.90, bottom=0.24,
    wspace=0.34
)

sub = pareto5[
    (pareto5["dataset"] == "IgG-all96") &
    (pareto5["feature"] == "onehot")
]

metrics = [
    ("recall_mean", "recall_sd", "Pareto Recall@K", "A", "IgG-96 Pareto recall"),
    ("hypervolume_mean", "hypervolume_sd", "Hypervolume", "B", "IgG-96 hypervolume"),
    ("igd_mean", "igd_sd", "IGD", "C", "IgG-96 IGD"),
]

for ax, (metric, err, ylabel, letter, title) in zip(axes, metrics):
    for model in MODEL_ORDER:
        g = sub[sub["model"] == model].sort_values("k")
        if g.empty:
            continue
        ax.errorbar(
            g["k"], g[metric],
            yerr=finite_err(g[err]) if err in g.columns else None,
            marker=MODEL_MARKERS.get(model, "o"),
            linestyle=MODEL_LINESTYLE.get(model, "-"),
            capsize=2,
            label=model,
        )
    ax.set_xlabel("Screening budget K")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{letter}  {title}", pad=8)
    clean_axes(ax)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=5,
    frameon=False,
)

save_figure(fig, "Supplementary_Figure_S1_IgG96_Pareto_Clean")
plt.show()

# Supplementary Figure S2 — Controlled attribution and latent robustness

The former crowded main-paper attribution figure is moved to the Supplement.  
Long A–H definitions are kept in the **caption**, not printed under the axes.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8.8, 5.8))
fig.subplots_adjust(
    left=0.08, right=0.985, top=0.94, bottom=0.16,
    wspace=0.34, hspace=0.46
)

# A/B — ranking/gap mutation ablation
for col, task in enumerate(["affinity", "ova"]):
    ax = axes[0, col]
    sub = loss_mut[
        (loss_mut["feature"] == "onehot") &
        (loss_mut["task"] == task)
    ].copy()
    sub["ord"] = sub["loss_arm"].map({a: j for j, a in enumerate(LOSS_ORDER)})
    sub = sub.sort_values("ord")
    x = np.arange(len(sub))

    ax.errorbar(
        x, sub["mcc_mean"],
        yerr=finite_err(sub["mcc_site_sd"]),
        marker="o", linestyle="-", capsize=2
    )
    ax.set_xticks(x)
    ax.set_xticklabels(
        [LOSS_SHORT.get(a, a) for a in sub["loss_arm"]],
        rotation=12, ha="right"
    )
    ax.set_ylabel("Mutation MCC")
    letter = "A" if task == "affinity" else "B"
    label = "Target loss ablation" if task == "affinity" else "OVA loss ablation"
    ax.set_title(f"{letter}  {label}", pad=8)
    clean_axes(ax)

# C — A-H mutation MCC
ax = axes[0, 2]
sub = component_mut[component_mut["feature"] == "onehot"].copy()
for task, marker in [("affinity", "o"), ("ova", "s")]:
    g = sub[sub["task"] == task].copy()
    g["letter"] = g["arm"].map(ARCH_ARM_LABELS)
    g["ord"] = g["letter"].map({c: j for j, c in enumerate("ABCDEFGH")})
    g = g.sort_values("ord")
    ax.errorbar(
        np.arange(len(g)), g["mcc_mean"],
        yerr=finite_err(g["mcc_site_sd"]),
        marker=marker, linestyle="-", capsize=2,
        label="Target proxy" if task == "affinity" else "OVA proxy",
    )
ax.set_xticks(range(8))
ax.set_xticklabels(list("ABCDEFGH"))
ax.set_ylabel("Mutation MCC")
ax.set_title("C  Architecture controls", pad=8)
clean_axes(ax)

# D/E — A-H ISO OneHot Pareto at K=20
arch = component_pareto[
    (component_pareto["suite"] == "architecture_8arm") &
    (component_pareto["dataset"] == "ISO") &
    (component_pareto["feature"] == "onehot") &
    (component_pareto["k"] == 20)
].copy()
arch["letter"] = arch["arm"].map(ARCH_ARM_LABELS)
arch["ord"] = arch["letter"].map({c: j for j, c in enumerate("ABCDEFGH")})
arch = arch.sort_values("ord")

for ax, metric, err, ylabel, letter, title in [
    (axes[1, 0], "recall_mean", "recall_sd", "Pareto Recall@20", "D", "ISO architecture recall"),
    (axes[1, 1], "igd_mean", "igd_sd", "IGD@20", "E", "ISO architecture IGD"),
]:
    x = np.arange(len(arch))
    ax.errorbar(
        x, arch[metric],
        yerr=finite_err(arch[err]),
        marker="o", linestyle="-", capsize=2,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(arch["letter"])
    ax.set_xlabel("Architecture arm")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{letter}  {title}", pad=8)
    clean_axes(ax)

# F — latent-PCA robustness
ax = axes[1, 2]
space_order = ["logits", "pca16", "pca32", "pca64"]
space_label = {
    "logits": "Logits",
    "pca16": "PCA16",
    "pca32": "PCA32",
    "pca64": "PCA64",
}
sub = latent[
    (latent["dataset"] == "ISO") &
    (latent["feature"] == "onehot")
]
for space in space_order:
    g = sub[sub["score_space"] == space].sort_values("k")
    if g.empty:
        continue
    ax.errorbar(
        g["k"], g["recall_mean"],
        yerr=finite_err(g["recall_sd"]),
        marker="o", linestyle="-", capsize=2,
        label=space_label[space],
    )
ax.set_xlabel("Screening budget K")
ax.set_ylabel("Pareto Recall@K")
ax.set_title("F  Latent-score robustness", pad=8)
ax.legend(frameon=False, fontsize=6, loc="best")
clean_axes(ax)

# Only the Target/OVA key is outside the panels.
proxy_handles = [
    plt.Line2D([], [], marker="o", linestyle="-", label="Target proxy"),
    plt.Line2D([], [], marker="s", linestyle="-", label="OVA proxy"),
]
fig.legend(
    handles=proxy_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=2,
    frameon=False,
)

save_figure(fig, "Supplementary_Figure_S2_Controlled_Attribution")
plt.show()

## Recommended captions

In [ ]:
captions = f"""
Figure 2. Mutation generalization is representation-sensitive and degrades with sequence novelty. (A,B) Primary top-residue mutation-holdout MCC across the three principal representations. (C,D) Paired Site-ESM2 minus Mean-ESM2 MCC differences; asterisks indicate Holm-corrected paired Wilcoxon p<0.05. (E,F) Mean-ESM2 mutation MCC stratified by minimum Hamming distance to the corresponding training support; error bars are 95% site-block bootstrap confidence intervals.

Figure 3. Cross-platform transfer and fixed-budget multi-objective prioritization. (A-D) Spearman correlations with continuous ISO and IgG-42 target- and OVA-binding measurements across sequence representations. Neural error bars show optimization-seed standard deviations. (E,F) ISO Mean-Fusion Pareto Recall@K and IGD under equal candidate budgets. Model ordering varies across endpoints, representations, and screening budgets.

{FIG4_CAPTION}

Supplementary Figure S1. IgG-all96 OneHot fixed-budget Pareto prioritization. Recall, hypervolume, and IGD are shown as functions of screening budget. Neural error bars represent optimization-seed standard deviations.

Supplementary Figure S2. Controlled attribution of auxiliary losses, sharing, routing, and latent representation. (A,B) Focal/ranking/gap mutation ablations within the shared Standard-MOLM architecture. (C) OneHot mutation MCC across the A-H architecture controls. (D,E) ISO OneHot fixed-budget Pareto recall and IGD at K=20 across the same controls. (F) Standard-MOLM output-logit and latent-PCA Recall@K. Architecture arms: A, shared base; B, shared + dominance; C, shared + dominance + PCGrad; D, shared + dominance + private adapters; E, Full Routed-MOLM; F, independent matched-loss; G, capacity-matched shared + dominance; H, capacity-matched shared + dominance + PCGrad.
""".strip()

caption_path = WORK_ROOT / "FIGURE_CAPTIONS.txt"
caption_path.write_text(captions, encoding="utf-8")
print(caption_path)
print(captions)

## 6. Reproducibility manifest and final ZIP

In [ ]:
manifest = {
    "purpose": "publication figure generation v2; no model retraining; existing manuscript Figure 1 preserved",
    "seeds": SEEDS,
    "diagnostic_mode": DIAGNOSTIC_MODE,
    "expected_config": EXPECTED_CONFIG,
    "resolved_roots": {
        "unified": str(UNIFIED_ROOT),
        "molm_st": str(MOLM_ST_ROOT),
        "component_pareto": str(COMPONENT_ROOT),
    },
    "figure_files": sorted([p.name for p in FIG_DIR.iterdir() if p.is_file()]),
    "source_data_files": sorted([p.name for p in SOURCE_DIR.iterdir() if p.is_file()]),
}

manifest_path = WORK_ROOT / "FIGURE_REPRODUCIBILITY_MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_base = WORK_ROOT.parent / "MOLM_Publication_Figures_v9"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=WORK_ROOT)

print("Manifest:", manifest_path)
print("Final ZIP:", zip_path)
print("Generated figures:")
for p in sorted(FIG_DIR.glob("*")):
    print(" ", p.name)

## Figure-to-paper compression plan

Recommended main-paper display strategy:

- **Figure 1:** keep the existing manuscript architecture figure.
- **Figure 2:** mutation holdout + Site-ESM2 + Hamming-distance generalization.
- **Figure 3:** cross-platform transfer + fixed-budget ISO Pareto behavior.
- **Figure 4:** updated diagnostic figure using the earlier paper's diagnostic concept, rebuilt with the current fixed-budget/latent-PCA analysis.
- **Supplementary Figure S1:** IgG-all96 Pareto regime.
- **Supplementary Figure S2:** detailed controlled attribution / loss / A-H component analysis.

This lets the Results text describe the qualitative pattern and point to figures, while exact values, confidence intervals, complete budgets, and complete ablation tables remain in the Supplement.
